<a href="https://colab.research.google.com/github/elliemci/agents/blob/main/agentic_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic RAG

## Required Libraries

In [ ]:
!pip install datasets langchain langchain-community openai smolagents chromadb

In [ ]:
!pip install rich

In [ ]:
!pip install hf_xet

## Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/ColabNotebooks/AgentsCourse

In [ ]:
from datasets import load_dataset
from langchain_community.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings #OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from smolagents import CodeAgent, Tool, HfApiModel

from langchain_community.embeddings import HuggingFaceEmbeddings

## Environment Varaibles

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["HUGGING_FACE_HUB_TOKEN"] = userdata.get('huggingface_hub_access_token')
os.environ["NEWS_API_KEY"] = userdata.get("NewsAPI_KEY")

## Tools

### Database Retrieval

1. Load MedMCQA

2. Prepare data: extract questions, contexts, and answers

3. Chunk the data

4. Embed and store in a vectorstore

5. Create a retriever tool

MedMCQA is a large-scale Multiple-Choice Question Answering dataset which contains real Medical exam Question Answering data set [medmcqa](https://huggingface.co/datasets/medmcqa) hast the follwing data fields:
* id: a string question identifier for each example
* question: a string text
* opa: Option A
* opb: Option B
* opc: Option D
* cop: Correct option
* choice_type {"single", "multi"}: question type, "single"-choice contains a single option, and "multi"-choice question contains a combination of multiple options
* exp: expert's explanation of the answer
* subject_name: medical subject name of thequestion
* topic_name: medical topic name

In [ ]:
# load MedMCQA from Huggingface
dataset = load_dataset("medmcqa", split="train") # for speed use "train[:1000]"

In [ ]:
dataset[1]

### Clean up NA data

In [ ]:
# handling missing values by converting to pandas df and using isna()
import pandas as pd

df = pd.DataFrame(dataset)
df.isna().sum()

In [ ]:
# drop rows with missing values in columns exp and topic_name
df = df.dropna(subset=["exp"])
print(df.isna().sum())
df = df.dropna(subset=["topic_name"])
print(df.isna().sum())

In [ ]:
# convert no missin values dataframe back to datasest
dataset = dataset.from_pandas(df)

In [ ]:
# combine question and explanation for context
docs = []

for item in dataset:
  content = f"Q: {item['question']}\nA) {item['opa']} B) {item['opb']} C) {item['opc']} D) {item['opd']}\nAnswer: {item['cop']}\nExplanation: {item.get('exp', '')}"
  docs.append(Document(page_content=content, metadata={"id": item['id']}))

### Medical Dataset Retrieval Tool

In [ ]:
def get_correct_answer_text(ex):
    """
    Given an example from MedMCQA, extract the actual text(s) of the correct answer(s).
    Works for both 'single' and 'multi' choice_type.
    """
    options = {
        "1": ex.get("opa", "").strip(),
        "2": ex.get("opb", "").strip(),
        "3": ex.get("opc", "").strip(),
        "4": ex.get("opd", "").strip()
    }

    # get the text from the correct answer or answers
    correct_option_raw = str(ex.get("cop", "")).strip()
    correct_indices = [opt.strip() for opt in correct_option_raw.split(",") if opt.strip()]

    # Fallback: infer choice_type from number of correct answers
    declared_type = ex.get("choice_type", "single").strip().lower()
    inferred_type = "multi" if len(correct_indices) > 1 else "single"

    # Use whichever is more accurate
    choice_type = inferred_type if declared_type not in {"single", "multi"} else declared_type

    correct_texts = [options.get(idx, f"[Unknown Option {idx}]") for idx in correct_indices]

    return correct_texts if choice_type == "multi" else correct_texts[0]


In [ ]:
ex = {
    "question": "What causes increased blood pressure?",
    "opa": "Low salt intake",
    "opb": "Dehydration",
    "opc": "High sodium levels",
    "opd": "Regular exercise",
    "cop": "2, 3",
    "choice_type": "multi",
    "exp": "High sodium levels cause water retention, increasing blood volume and pressure.",
    "subject_name": "Physiology",
    "topic_name": "Cardiovascular System"
}

print(get_correct_answer_text(ex))

In [ ]:
import shutil
from tqdm import tqdm

# setup with embeding-based retriever sentence-transformer
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = []

# utility to get correct answer text
def get_correct_answer_text(ex):
    options = {
        "1": ex.get("opa", "").strip(),
        "2": ex.get("opb", "").strip(),
        "3": ex.get("opc", "").strip(),
        "4": ex.get("opd", "").strip()
    }

    correct_option_raw = str(ex.get("cop", "")).strip()
    correct_indices = [opt.strip() for opt in correct_option_raw.split(",") if opt.strip()]

    # fallback or check on consistency
    declared_type = ex.get("choice_type", "single").strip().lower()
    inferred_type = "multi" if len(correct_indices) > 1 else "single"
    choice_type = inferred_type if declared_type not in {"single", "multi"} else declared_type

    correct_texts = [options.get(idx, f"[Unknown Option {idx}]") for idx in correct_indices]
    return correct_texts if choice_type == "multi" else correct_texts[0]

# Tool definition
class MedDataChromaRetrieverTool(Tool):
    name = "med_data_chroma_retrieve"
    description = "Retrieves correct answers from MedMCQA using semantic similarity (Chroma vector store)."
    inputs = {
        "query": {
            "type": "string",
            "description": "The medical question you want to search."
        }
    }
    output_type = "string"

    def __init__(self, persist_directory="medmcqa_chromadb"):
        self.is_initialized = False
        self.vectorstore = None
        self.persist_directory = persist_directory

        if os.path.exists(persist_directory):
            self.vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embedding_model)
            self.is_initialized = True
        else:
            os.makedirs(self.persist_directory, exist_ok=True)
            self._build_vector_index()

    def _build_vector_index(self):
        med_dataset = load_dataset("medmcqa", split="train[:1000]")

        for ex in tqdm(med_dataset, desc="Building vector index for MedMCQA dataset"):
            correct_answer_text = get_correct_answer_text(ex)
            explanation = str(ex.get("exp", "") or "").strip()
            subject = str(ex.get("subject_name", "") or "").strip()
            topic = str(ex.get("topic_name", "") or "").strip()

            content = f"""Question: {ex['question']}
Correct Answer: {correct_answer_text}"""
            if explanation:
                content += f"\nExplanation: {explanation}"
            if subject and subject.upper() != "NA":
                content += f"\nSubject: {subject}"
            if topic and topic.upper() != "NA":
                content += f"\nTopic: {topic}"

            metadata = {
                "subject": subject,
                "topic": topic,
                "answer": correct_answer_text
            }

            split_docs = text_splitter.create_documents([content], metadatas=[metadata])
            docs.extend(split_docs)

        if os.path.exists(self.persist_directory):
            shutil.rmtree(self.persist_directory)

        self.vectorstore = Chroma.from_documents(
            docs,
            embedding=embedding_model,
            persist_directory=self.persist_directory
        )
        self.vectorstore.persist()
        self.is_initialized = True

    def forward(self, query: str):
        if not self.is_initialized:
            return "Vector store not initialized."

        results = self.vectorstore.similarity_search(query, k=3)
        return "\n\n".join([doc.page_content for doc in results])

# initialize the medical data retrieval tool
med_data_chroma_retrieve_tool = MedDataChromaRetrieverTool()


### Add Conversation Memory

### Web Search

For **Hybrid RAG** and  Web Seach use dynamic Tool selection with ToolCallingAgent instead of CodeAgent, guide the LLM's behavior with tool descriptions or system prompts.

In [ ]:
from smolagents import ToolCallingAgent, DuckDuckGoSearchTool

# Initialize model and tool
model = HfApiModel()

med_data_chroma_retrieve_tool.description = (
    "First try this tool. Retrieves accurate answers from the MedMCQA medical dataset. "
    "Use this for any standard medical knowledge, symptoms, treatments, or explanations."
)

web_tool = DuckDuckGoSearchTool()
web_tool.description = (
    "Use this only if the question cannot be answered from the medical dataset. "
    "Good for the latest or web-only info like new treatments, breaking news, or current guidelines."
)


### Hub Stats Tool

In [ ]:
from smolagents import Tool
from huggingface_hub import list_models, model_info

from rich import print
from IPython.display import Markdown

class HubTopModelByTaskTool(Tool):
    """
    Fetches the most downloaded Hugging Face model used for a given task or in a field like finanial investments, medical, etc.
    """
    name = "hub_top_model_by_task"
    description = "Fetches the most downloaded Hugging Face model used for a given task or in a field like finanial investments, medical, etc."
    inputs = {
        "task": {
            "type": "string",
            "description": "The model name of a top performing model in a given field of interest."
        }
    }
    output_type = "string"

    def forward(self, task: str):
        try:
            # search and list models matching the task or keyword sorted by downloads
            models = list(list_models(search=task, sort="downloads", direction=-1, limit=1))

            if not models:
                return f"No models found for task '{task}'. Try something broader."

            top_model = models[0]
            info = model_info(top_model.id)

            description = info.cardData.get("summary") if info.cardData else None

            return (
                # emoji picker Ctrl + Cmd + Space
                f"Most Downloaded Model for [bold]'{task}':[/bold]\n"
                f"Model ID: [bold]{top_model.id}[/bold]\n"
                f"Downloads: [bold]{top_model.downloads:,}[/bold]\n"
                f"Description: {description or 'No description available.'}\n"
                f"View on Hugging Face: https://huggingface.co/{top_model.id}"
            )
        except Exception as e:
            return f"Error while searching models for '{task}': {str(e)}"

# Initialize the tool
hub_top_model_by_task_tool = HubTopModelByTaskTool()


In [ ]:
# example usage: Get the most downloaded model used in medical research
print(hub_top_model_by_task_tool("medical research"))

### Latest News on a topic Tool

In [ ]:
from smolagents import Tool
import os
import requests

class LatestNewsTool(Tool):
    name = "latest_news"
    description = "Fetches the latest news headlines on a specific topic."
    inputs = {
        "topic": {
            "type": "string",
            "description": "The topic or keywords for the latest news e.g., AI, economy, employment."
        }
    }
    output_type = "string"

    def __init__(self):
        super().__init__()
        self.api_key = os.environ.get("NEWS_API_KEY")
        self.base_url = "https://newsapi.org/v2/everything"
        self.is_initialized = True

        if not self.api_key:
            raise ValueError("NEWS_API_KEY environment variable not found. Set it before using this tool.")

    def forward(self, topic: str) -> str:
        try:
            print(f"Fetching news for topic: {topic}")

            params = {
                "q": topic,
                "apiKey": self.api_key,
                "language": "en",
                "sortBy": "publishedAt",
                "pageSize": 5
            }

            response = requests.get(self.base_url, params=params)
            response.raise_for_status()
            data = response.json()

            if not data.get("articles"):
                return f"No recent news found for topic: {topic}"

            results = []
            for article in data["articles"]:
                title = article.get("title", "No title")
                url = article.get("url", "")
                source = article.get("source", {}).get("name", "Unknown source")
                results.append(f"{title}\n{url} (Source: {source})")

            return "\n\n".join(results)

        except Exception as e:
            return f"Error fetching news: {str(e)}"


news_tool = LatestNewsTool()


In [ ]:
# test news tool
print(news_tool("Humanoid robots"))

In [ ]:
model = HfApiModel()

agent = CodeAgent(
    tools=[med_data_chroma_retrieve_tool, web_tool, hub_top_model_by_task_tool],
    model=model
)

In [ ]:
response = agent("What is the latest news on artificial intelligence?")
print(response)